# 07 — Spark Structured Streaming Consumer
**Role: Pipeline & Visualization Engineer**

Reads the `grab_reviews` Kafka topic in real-time, loads the trained Naive Bayes model, runs sentiment inference, then sinks results to **Elasticsearch**.

**Pipeline:** Kafka `grab_reviews` → Spark Streaming → Elasticsearch `grab_sentiment`

## 1. Install dependencies

In [ ]:
import subprocess, sys

pkgs = ['pyspark==3.4.2', 'elasticsearch==8.13.0', 'scikit-learn', 'kafka-python', 'pandas', 'joblib']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])

KAFKA_SPARK_PACKAGE = 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.2'
print('Dependencies ready.')
print(f'Spark Kafka package: {KAFKA_SPARK_PACKAGE}')

## 2. Configuration

In [ ]:
import os

# ── Kafka ──────────────────────────────────────────────────────────────────
KAFKA_BOOTSTRAP = os.getenv('KAFKA_BOOTSTRAP', 'localhost:9092')
KAFKA_TOPIC     = 'grab_reviews'

# ── Elasticsearch ──────────────────────────────────────────────────────────
ES_HOST  = os.getenv('ES_HOST', 'localhost')
ES_PORT  = int(os.getenv('ES_PORT', 9200))
ES_INDEX = 'grab_sentiment'
ES_USER  = os.getenv('ES_USER', 'elastic')
ES_PASS  = os.getenv('ES_PASS', 'changeme')

# ── Model files (saved by NLP engineer in notebooks 02/03) ────────────────
# Priority 1: Naive Bayes (separate vectorizer + classifier files)
NB_VECTORIZER = None
NB_MODEL      = None

NB_VECTORIZER_PATHS = ['models/tfidf_vectorizer.joblib', '../models/tfidf_vectorizer.joblib']
NB_MODEL_PATHS      = ['models/naive_bayes.joblib',      '../models/naive_bayes.joblib']

for p in NB_VECTORIZER_PATHS:
    if os.path.exists(p): NB_VECTORIZER = p; break
for p in NB_MODEL_PATHS:
    if os.path.exists(p): NB_MODEL = p; break

# Priority 2: sklearn Pipeline saved as .pkl
PIPELINE_PKL = None
for p in ['models/best_model.pkl', '../models/best_model.pkl',
          'models/sentiment_model.pkl', '../models/sentiment_model.pkl']:
    if os.path.exists(p): PIPELINE_PKL = p; break

# ── Spark checkpoint ────────────────────────────────────────────────────────
CHECKPOINT_DIR = '/tmp/grab_spark_checkpoint'

# ── Data file (for batch comparison) ───────────────────────────────────────
DATA_FILE = 'cleaned_data (2).csv'
for candidate in ['cleaned_data (2).csv', 'cleaned_data.csv',
                  '../cleaned_data (2).csv', '../cleaned_data.csv']:
    if os.path.exists(candidate): DATA_FILE = candidate; break

print(f'Kafka      : {KAFKA_BOOTSTRAP}  topic={KAFKA_TOPIC}')
print(f'ES         : {ES_HOST}:{ES_PORT}  index={ES_INDEX}')
print(f'Vectorizer : {NB_VECTORIZER}')
print(f'NB model   : {NB_MODEL}')
print(f'Pipeline   : {PIPELINE_PKL}')
print(f'Data file  : {DATA_FILE}')

## 3. Load the trained model

In [ ]:
import joblib, pickle, os

LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

class NBPipeline:
    """Wraps TF-IDF vectorizer + Naive Bayes classifier into a single predict() call."""
    def __init__(self, vectorizer, classifier):
        self.vectorizer  = vectorizer
        self.classifier  = classifier

    def predict(self, texts):
        X = self.vectorizer.transform(texts)
        return self.classifier.predict(X)

# ── Try loading Naive Bayes (vectorizer + classifier) first ───────────────
model_pipeline = None
model_source   = None

if NB_VECTORIZER and NB_MODEL:
    vectorizer = joblib.load(NB_VECTORIZER)
    classifier = joblib.load(NB_MODEL)
    model_pipeline = NBPipeline(vectorizer, classifier)
    model_source   = f'Naive Bayes  ({NB_MODEL} + {NB_VECTORIZER})'

# ── Fall back to a saved sklearn Pipeline pkl ─────────────────────────────
elif PIPELINE_PKL:
    with open(PIPELINE_PKL, 'rb') as f:
        model_pipeline = pickle.load(f)
    model_source = f'Pipeline pkl  ({PIPELINE_PKL})'

else:
    raise FileNotFoundError(
        'No trained model found.\n'
        'Expected one of:\n'
        '  models/naive_bayes.joblib  +  models/tfidf_vectorizer.joblib\n'
        '  models/best_model.pkl\n'
        'Run notebooks 01–02 first to train and save the Naive Bayes model.'
    )

print(f'Model loaded: {model_source}')
test = model_pipeline.predict(['good service fast delivery'])
label = LABEL_MAP.get(int(test[0]), str(test[0])) if isinstance(test[0], (int, float)) else str(test[0])
print(f'Sanity check: "good service fast delivery" → {label}')

## 4. Start Spark session

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('GrabSentimentStreaming')
    .config('spark.jars.packages', KAFKA_SPARK_PACKAGE)
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.streaming.stopGracefullyOnShutdown', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark session started:', spark.version)

## 5. Read stream from Kafka

In [ ]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

review_schema = StructType([
    StructField('review_id',          StringType(),  True),
    StructField('username',           StringType(),  True),
    StructField('timestamp',          StringType(),  True),
    StructField('final_clean_text',   StringType(),  True),
    StructField('score',              IntegerType(), True),
    StructField('sentiment_category', StringType(),  True),
    StructField('sentiment_label',    IntegerType(), True),
    StructField('confidence_level',   StringType(),  True),
    StructField('app_version',        StringType(),  True),
    StructField('thumbs_up_count',    IntegerType(), True),
    StructField('word_count',         IntegerType(), True),
    StructField('has_reply',          BooleanType(), True),
    StructField('ingested_at',        StringType(),  True),
])

raw_stream = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP)
    .option('subscribe', KAFKA_TOPIC)
    .option('startingOffsets', 'earliest')
    .option('maxOffsetsPerTrigger', 200)
    .load()
)

parsed = (
    raw_stream
    .select(
        col('key').cast('string').alias('kafka_key'),
        col('offset').alias('kafka_offset'),
        col('partition').alias('kafka_partition'),
        from_json(col('value').cast('string'), review_schema).alias('data')
    )
    .select('kafka_key', 'kafka_offset', 'kafka_partition', 'data.*')
)

parsed.printSchema()

## 6. UDF — apply Naive Bayes model

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import pickle, joblib, io

# Broadcast the model as bytes so each Spark worker gets its own copy
model_bytes_broadcast = spark.sparkContext.broadcast(
    pickle.dumps(model_pipeline)   # NBPipeline is picklable
)

@udf(returnType=StringType())
def predict_sentiment(text):
    if text is None or str(text).strip() == '':
        return 'Neutral'
    try:
        m = pickle.loads(model_bytes_broadcast.value)
        pred = m.predict([str(text)])[0]
        label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
        if isinstance(pred, (int, float)):
            return label_map.get(int(pred), str(pred))
        return str(pred)
    except Exception as e:
        return f'Error: {e}'

enriched = (
    parsed
    .withColumn('predicted_sentiment', predict_sentiment(col('final_clean_text')))
    .withColumn('processed_at', current_timestamp())
)

print('Enriched schema:')
enriched.printSchema()

## 7. Elasticsearch sink

In [ ]:
from elasticsearch import Elasticsearch, helpers

def get_es_client():
    return Elasticsearch(
        f'http://{ES_HOST}:{ES_PORT}',
        basic_auth=(ES_USER, ES_PASS),
        verify_certs=False,
        request_timeout=30
    )

def create_es_index(es):
    if es.indices.exists(index=ES_INDEX):
        return
    mapping = {
        'mappings': {'properties': {
            'review_id':           {'type': 'keyword'},
            'username':            {'type': 'keyword'},
            'timestamp':           {'type': 'date', 'format': 'MM/dd/yyyy H:mm||yyyy-MM-dd HH:mm:ss||epoch_millis'},
            'final_clean_text':    {'type': 'text', 'analyzer': 'english'},
            'score':               {'type': 'integer'},
            'sentiment_category':  {'type': 'keyword'},
            'predicted_sentiment': {'type': 'keyword'},
            'app_version':         {'type': 'keyword'},
            'thumbs_up_count':     {'type': 'integer'},
            'word_count':          {'type': 'integer'},
            'has_reply':           {'type': 'boolean'},
            'ingested_at':         {'type': 'date'},
            'processed_at':        {'type': 'date'},
            'kafka_partition':     {'type': 'integer'},
            'kafka_offset':        {'type': 'long'},
        }},
        'settings': {'number_of_shards': 1, 'number_of_replicas': 0}
    }
    es.indices.create(index=ES_INDEX, body=mapping)
    print(f'Index "{ES_INDEX}" created.')

def write_batch_to_es(batch_df, batch_id):
    rows = batch_df.collect()
    if not rows:
        return
    es = get_es_client()
    create_es_index(es)
    actions = []
    for row in rows:
        doc = row.asDict()
        for k, v in doc.items():
            if hasattr(v, 'isoformat'): doc[k] = v.isoformat()
        actions.append({'_index': ES_INDEX, '_id': doc.get('review_id'), '_source': doc})
    ok, errs = helpers.bulk(es, actions, raise_on_error=False)
    print(f'Batch {batch_id:>4}: indexed {ok} docs  |  errors: {len(errs)}')

print('ES sink functions ready.')

## 8. Start the streaming query

In [ ]:
import shutil, os

if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print(f'Cleared checkpoint: {CHECKPOINT_DIR}')

query = (
    enriched.writeStream
    .foreachBatch(write_batch_to_es)
    .option('checkpointLocation', CHECKPOINT_DIR)
    .trigger(processingTime='5 seconds')
    .start()
)

print(f'Streaming query started  (id={query.id})')
print('Consuming from Kafka → Elasticsearch ...')

## 9. Monitor progress

In [ ]:
import time

MAX_WAIT       = 600
CHECK_INTERVAL = 15
elapsed        = 0

while query.isActive and elapsed < MAX_WAIT:
    prog = query.lastProgress
    if prog:
        rows_in  = prog.get('numInputRows', 0)
        rows_sec = prog.get('inputRowsPerSecond', 0.0)
        proc_sec = prog.get('processedRowsPerSecond', 0.0)
        print(f'[{elapsed:>4}s]  rows={rows_in}  in={rows_sec:.1f}/s  proc={proc_sec:.1f}/s')
    else:
        print(f'[{elapsed:>4}s]  Waiting for first micro-batch...')
    time.sleep(CHECK_INTERVAL)
    elapsed += CHECK_INTERVAL

query.stop()
print('\nStreaming query stopped.')

## 10. Batch mode comparison

In [ ]:
import time, pandas as pd

df_batch = pd.read_csv(DATA_FILE)
print(f'Batch mode: {len(df_batch):,} rows from {DATA_FILE}')

texts = df_batch['final_clean_text'].fillna('').tolist()

t0 = time.time()
preds = model_pipeline.predict(texts)
batch_elapsed = time.time() - t0

df_batch['predicted_sentiment'] = [
    LABEL_MAP.get(int(p), str(p)) if isinstance(p, (int, float)) else str(p)
    for p in preds
]

rows_per_sec = len(df_batch) / batch_elapsed
print(f'Batch inference: {len(df_batch):,} rows in {batch_elapsed:.2f}s → {rows_per_sec:.0f} rows/s')

In [ ]:
# Index batch results to a separate ES index
BATCH_INDEX = 'grab_sentiment_batch'
es = get_es_client()
if not es.indices.exists(index=BATCH_INDEX):
    es.indices.create(index=BATCH_INDEX,
        body={'settings': {'number_of_shards': 1, 'number_of_replicas': 0}})

actions = [
    {'_index': BATCH_INDEX, '_id': str(row.get('review_id', i)), '_source': row.to_dict()}
    for i, (_, row) in enumerate(df_batch.iterrows())
]
t1 = time.time()
ok, errs = helpers.bulk(es, actions, chunk_size=500, raise_on_error=False)
idx_elapsed = time.time() - t1
print(f'Batch ES index: {ok} docs in {idx_elapsed:.2f}s  |  errors: {len(errs)}')

In [ ]:
import pandas as pd

pd.DataFrame({
    'Mode':              ['Batch',                        'Streaming (Spark)'],
    'Rows processed':    [len(df_batch),                  len(df_batch)],
    'Inference time':    [f'{batch_elapsed:.2f}s',        '5s micro-batches'],
    'Throughput':        [f'{rows_per_sec:.0f} rows/s',   'Kafka-bounded'],
    'Latency':           ['High (runs once)',              'Low (~5s per batch)'],
    'Use case':          ['Historical reports',            'Real-time dashboards'],
})

---
### Summary
| Component | Detail |
|-----------|--------|
| Source | Kafka `grab_reviews` |
| Model | `models/naive_bayes.joblib` + `models/tfidf_vectorizer.joblib` |
| Sink | Elasticsearch `grab_sentiment` |
| Batch index | `grab_sentiment_batch` |
| Next | `08_visualization.ipynb` |